In [ ]:
import numpy as np
import pandas as pd

# Import Dataset

## Download competition

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls /content/drive/MyDrive/Projects/LLMClassificationFinetuning/

In [ ]:
base = "/content/drive/MyDrive/Projects/LLMClassificationFinetuning/"
train_file_name = "train.csv"
train_path = f"{base}/{train_file_name}"
train = pd.read_csv(train_path)

test_file_name = "test.csv"
test_path = f"{base}/{train_file_name}"
test = pd.read_csv(test_path)

# Work with BERT

## Import Bert

In [ ]:
import torch
import torch.nn.functional as F
import os
from transformers import AutoTokenizer, BertForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

weights_path = f"{base}/weights"
weights_files = os.listdir(path=weights_path)
bert_path = "bert-base-uncased"

if "tokenizer.json" not in weights_files:
    tokenizer = AutoTokenizer.from_pretrained(bert_path)
else:
    tokenizer = AutoTokenizer.from_pretrained(weights_path)

if "model.safetensors" not in weights_files:
    model = BertForSequenceClassification.from_pretrained(bert_path)
else:
    model = BertForSequenceClassification.from_pretrained(weights_path)

In [ ]:
from torch.utils.data import Dataset, DataLoader


class TextPairDataset(Dataset):
    def __init__(self, dataframe):
        self.prompts = []
        self.responses = []
        self.labels = []

        for response_index, response_content in train.iterrows():

            prompt = response_content["prompt"].strip("\"[]")
            self.prompts.append(prompt)
            self.prompts.append(prompt)

            response_a = response_content["response_a"]
            response_b = response_content["response_b"]
            self.responses.append(response_a.strip("\"[]"))
            self.responses.append(response_b.strip("\"[]"))

            if response_content["winner_tie"]:
                self.labels.append(0)
                self.labels.append(0)
            elif response_content["winner_model_a"]:
                self.labels.append(1)
                self.labels.append(0)
            else:
                self.labels.append(0)
                self.labels.append(1)

    def __len__(self):
        return len(self.prompts)

    def __getitem__(self, idx):
        return self.prompts[idx], self.responses[idx], self.labels[idx]


def collate_fn(batch):
    prompts, responses, labels = zip(*batch)

    encoded = tokenizer(
        list(prompts),
        list(responses),
        padding=True,
        truncation=True,
        max_lenght=128,
        return_tensors="pt"
    )

    encoded["labels"] = torch.tensor(labels, dtype=torch.long)
    return encoded

train_dataset = TextPairDataset(train)
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)

model.to(device)
model.train()

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

for batch in train_loader:
    optimizer.zero_grad()
    batch = {k: v.to(device) for k, v in batch.items()}
    outputs = model(**batch)
    loss = outputs.loss
    loss.backward()
    optimizer.step()


In [ ]:
save_weight_directory = f"{base}/weights/"
model.save_pretrained(save_weight_directory)
tokenizer.save_pretrained(save_weight_directory)
print(f"Модель успешно сохранена в папку {save_weight_directory}")

## Make bert predictions

In [ ]:
class TestPairDataset(Dataset):
    def __init__(self, dataframe):
        self.prompts = []
        self.responses = []

        for response_index, response_content in train.iterrows():

            prompt = response_content["prompt"].strip("\"[]")
            self.prompts.append(prompt)
            self.prompts.append(prompt)

            response_a = response_content["response_a"]
            response_b = response_content["response_b"]
            self.responses.append(response_a.strip("\"[]"))
            self.responses.append(response_b.strip("\"[]"))

    def __len__(self):
        return len(self.prompts)

    def __getitem__(self, idx):
        return self.prompts[idx], self.responses[idx]


def collate_fn_for_test_dataset(batch):
    prompts, responses = zip(*batch)

    encoded = tokenizer(
        list(prompts),
        list(responses),
        padding=True,
        truncation=True,
        max_lenght=128,
        return_tensors="pt"
    )

    return encoded


test_dataset = TestPairDataset(test)
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    collate_fn=collate_fn_for_test_dataset
)


In [ ]:
model.eval()
model.to(device)

predictions = []
with torch.no_grad():
    for test_batch in test_loader:
        test_batch = {k: v.to(device) for k, v in test_batch.items()}
        test_outputs = model(**test_batch)
        logits = test_outputs.logits
        probabilities = F.softmax(logits, dim=-1)
        true_probs = probabilities[:, 1].cpu().tolist()
        predictions += true_probs

## Prepare Bert prediction submission

In [ ]:
submission = {}

for index, row_content in test.iterrows():
    identifier = row_content["id"]
    submission["id"] = submission.get("id", []) + [identifier]
    response_a_prob = predictions[index * 2]
    response_b_prob = predictions[index * 2 + 1]

    if response_a_prob - response_b_prob > 0.01:
        submission["winner_model_a"] = submission.get("winner_model_a", []) + [1]
        submission["winner_model_b"] = submission.get("winner_model_b", []) + [0]
        submission["winner_tie"] = submission.get("winner_tie", []) + [0]

    elif response_b_prob - response_a_prob > 0.01:
        submission["winner_model_a"] = submission.get("winner_model_a", []) + [0]
        submission["winner_model_b"] = submission.get("winner_model_b", []) + [1]
        submission["winner_tie"] = submission.get("winner_tie", []) + [0]

    else:
        submission["winner_model_a"] = submission.get("winner_model_a", []) + [0]
        submission["winner_model_b"] = submission.get("winner_model_b", []) + [0]
        submission["winner_tie"] = submission.get("winner_tie", []) + [1]


submission_df = pd.DataFrame.from_dict(submission)
submission_df.to_csv(f"{base}submission_bert.csv", index=False)


# Work with GPT

## Import GPT

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, OpenAIGPTForSequenceClassification

gpt_path = "openai-gpt"

tokenizer = AutoTokenizer.from_pretrained(gpt_path)
model = OpenAIGPTForSequenceClassification.from_pretrained(gpt_path)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

for batch in train_loader:
    optimizer.zero_grad()
    batch = {k: v.to(device) for k, v in batch.items()}
    outputs = model(**batch)
    loss = outputs.loss
    loss.backward()
    optimizer.step()

In [ ]:
model.eval()
model.to(device)

predictions = []
with torch.no_grad():
    for test_batch in test_loader:
        test_batch = {k: v.to(device) for k, v in test_batch.items()}
        test_outputs = model(**test_batch)
        logits = test_outputs.logits
        probabilities = F.softmax(logits, dim=-1)
        true_probs = probabilities[:, 1].cpu().tolist()
        predictions += true_probs

In [ ]:
submission = {}

for index, row_content in test.iterrows():
    identifier = row_content["id"]
    submission["id"] = submission.get("id", []) + [identifier]
    response_a_prob = predictions[index * 2]
    response_b_prob = predictions[index * 2 + 1]

    if response_a_prob - response_b_prob > 0.01:
        submission["winner_model_a"] = submission.get("winner_model_a", []) + [1]
        submission["winner_model_b"] = submission.get("winner_model_b", []) + [0]
        submission["winner_tie"] = submission.get("winner_tie", []) + [0]

    elif response_b_prob - response_a_prob > 0.01:
        submission["winner_model_a"] = submission.get("winner_model_a", []) + [0]
        submission["winner_model_b"] = submission.get("winner_model_b", []) + [1]
        submission["winner_tie"] = submission.get("winner_tie", []) + [0]

    else:
        submission["winner_model_a"] = submission.get("winner_model_a", []) + [0]
        submission["winner_model_b"] = submission.get("winner_model_b", []) + [0]
        submission["winner_tie"] = submission.get("winner_tie", []) + [1]


submission_df = pd.DataFrame.from_dict(submission)
submission_df.to_csv(f"{base}submission_gpt.csv", index=False)
